In [ ]:
# Portable data paths; execute notebooks from the package root.
import os
from pathlib import Path
_review_base = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
DATA_ROOT = Path(os.environ.get("REVIEW_DATA_ROOT", str(_review_base / "data"))).resolve()
RESULTS_ROOT = Path(os.environ.get("REVIEW_RESULTS_ROOT", str(_review_base / "results"))).resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


# 1. Aggregate hourly renewable generation by province

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DTYPE = np.float32
TIME_COL = "time_bj"

province_dict = {
    '\u5317\u4eac\u5e02': "beijing",
    '\u5929\u6d25\u5e02': "tianjin",
    '\u6cb3\u5317\u7701': "hebei",
    '\u5c71\u897f\u7701': "shanxi",
    '\u5185\u8499\u53e4\u81ea\u6cbb\u533a': "neimeng",
    '\u8fbd\u5b81\u7701': "liaoning",
    '\u5409\u6797\u7701': "jilin",
    '\u9ed1\u9f99\u6c5f\u7701': "heilongjiang",
    '\u4e0a\u6d77\u5e02': "shanghai",
    '\u6c5f\u82cf\u7701': "jiangsu",
    '\u6d59\u6c5f\u7701': "zhejiang",
    '\u5b89\u5fbd\u7701': "anhui",
    '\u798f\u5efa\u7701': "fujian",
    '\u6c5f\u897f\u7701': "jiangxi",
    '\u5c71\u4e1c\u7701': "shandong",
    '\u6cb3\u5357\u7701': "henan",
    '\u6e56\u5317\u7701': "hubei",
    '\u6e56\u5357\u7701': "hunan",
    '\u5e7f\u4e1c\u7701': "guangdong",
    '\u5e7f\u897f\u58ee\u65cf\u81ea\u6cbb\u533a': "guangxi",
    '\u6d77\u5357\u7701': "hainan",
    '\u91cd\u5e86\u5e02': "chongqing",
    '\u56db\u5ddd\u7701': "sichuan",
    '\u8d35\u5dde\u7701': "guizhou",
    '\u4e91\u5357\u7701': "yunnan",
    '\u897f\u85cf\u81ea\u6cbb\u533a': "xizang",
    '\u9655\u897f\u7701': "shannxi",
    '\u7518\u8083\u7701': "gansu",
    '\u9752\u6d77\u7701': "qinghai",
    '\u5b81\u590f\u56de\u65cf\u81ea\u6cbb\u533a': "ningxia",
    '\u65b0\u7586\u7ef4\u543e\u5c14\u81ea\u6cbb\u533a': "xinjiang",
}
PROVINCES_CN_ORDER = list(province_dict.keys())


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_county_matrix(parquet_fp: Path, return_time: bool = False):
    """Read county-level time series from parquet."""
    df = pd.read_parquet(parquet_fp)
    cols = [c for c in df.columns if c != TIME_COL]

    def _to_idx_name(c: str):
        idx, name = c.split("_", 1)
        return int(idx), name

    cols_sorted = sorted(cols, key=lambda c: _to_idx_name(c)[0])
    mat = df[cols_sorted].to_numpy(dtype=DTYPE, copy=False) / 1e6  # MW -> TW

    if return_time:
        time_vals = pd.to_datetime(df[TIME_COL])
        return mat, cols_sorted, time_vals
    return mat, cols_sorted


def read_county_to_province(mapping_fp: Path) -> dict[str, str]:
    """Read county to province mapping."""
    df = pd.read_csv(mapping_fp, dtype=str)
    required = {"county_key", "province_name"}
    if not required.issubset(df.columns):
        raise ValueError(f"{mapping_fp} must contain columns: {required}")
    return dict(zip(df["county_key"], df["province_name"]))


def aggregate_county_to_province(
    solar_county: np.ndarray,
    wind_county: np.ndarray,
    county_keys: list[str],
    county_to_province: dict[str, str],
) -> tuple[np.ndarray, np.ndarray]:
    """Aggregate county hourly generation to province hourly generation."""
    t, _ = solar_county.shape
    n_prov = len(PROVINCES_CN_ORDER)
    prov_idx = {p: i for i, p in enumerate(PROVINCES_CN_ORDER)}

    solar_prov = np.zeros((t, n_prov), dtype=DTYPE)
    wind_prov = np.zeros((t, n_prov), dtype=DTYPE)

    for j, county_key in enumerate(county_keys):
        prov = county_to_province.get(county_key)
        if prov is None or prov not in prov_idx:
            continue
        k = prov_idx[prov]
        solar_prov[:, k] += solar_county[:, j]
        wind_prov[:, k] += wind_county[:, j]

    return solar_prov, wind_prov


def main():
    repo_root = Path(str(DATA_ROOT / '6_Province_level_analysis/1_renewable_generation/data/china_solar_wind_complementarity'))

    solar_fp = repo_root / "data" / "county_generation_timeseries" / "china_county_solar_generation_timeseries.parquet"
    wind_fp = repo_root / "data" / "county_generation_timeseries" / "china_county_wind_generation_timeseries.parquet"
    county_map_fp = repo_root / "data" / "province_analysis_inputs" / "county_to_province.csv"

    out_dir = ensure_dir(repo_root / "outputs" / "hourly_generation")

    print(f"[INFO] Solar input: {solar_fp}")
    print(f"[INFO] Wind input: {wind_fp}")
    print(f"[INFO] Output dir: {out_dir}")

    # Read county generation
    solar_county, county_keys, time_vals = read_county_matrix(solar_fp, return_time=True)
    wind_county, _ = read_county_matrix(wind_fp, return_time=False)

    # Keep all 8760 hours (no alignment needed for generation data only)
    time_vals = pd.to_datetime(time_vals)

    county_to_province = read_county_to_province(county_map_fp)

    # Aggregate to province level
    solar_prov, wind_prov = aggregate_county_to_province(
        solar_county, wind_county, county_keys, county_to_province
    )

    # Create DataFrames with proper datetime index
    time_index = time_vals.reset_index(drop=True)
    province_names_en = [province_dict[p] for p in PROVINCES_CN_ORDER]

    # Solar generation DataFrame (TW)
    solar_df = pd.DataFrame(
        solar_prov * 1e3,  # TW -> GW
        index=time_index,
        columns=province_names_en
    )
    solar_df.index.name = "datetime"

    # Wind generation DataFrame (TW)
    wind_df = pd.DataFrame(
        wind_prov * 1e3,  # TW -> GW
        index=time_index,
        columns=province_names_en
    )
    wind_df.index.name = "datetime"

    # Save to CSV and XLSX
    solar_csv = out_dir / "province_hourly_solar_generation.csv"
    wind_csv = out_dir / "province_hourly_wind_generation.csv"
    solar_xlsx = out_dir / "province_hourly_solar_generation.xlsx"
    wind_xlsx = out_dir / "province_hourly_wind_generation.xlsx"

    solar_df.to_csv(solar_csv, encoding="utf-8-sig")
    wind_df.to_csv(wind_csv, encoding="utf-8-sig")
    solar_df.to_excel(solar_xlsx, engine="openpyxl")
    wind_df.to_excel(wind_xlsx, engine="openpyxl")

    print(f"[OK] Wrote: {solar_csv}")
    print(f"[OK] Wrote: {wind_csv}")
    print(f"[OK] Wrote: {solar_xlsx}")
    print(f"[OK] Wrote: {wind_xlsx}")

    # Print summary
    print(f"\n=== Summary ===")
    print(f"Time range: {time_index.min()} to {time_index.max()}")
    print(f"Total hours: {len(time_index)} (full year 2022)")
    print(f"Provinces: {len(province_names_en)}")
    print(f"\nAnnual generation (TWh):")
    for prov in province_names_en:
        solar_twh = solar_df[prov].sum() / 1e3  # GW*h -> TWh
        wind_twh = wind_df[prov].sum() / 1e3
        print(f"  {prov}: Solar={solar_twh:.2f}, Wind={wind_twh:.2f}")


if __name__ == "__main__":
    main()


# 2. Estimate hourly renewable curtailment

In [ ]:
'Estimate hourly wind and solar curtailment from generation, load, and reported utilization.'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import os
import sys


BASE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed')
GENERATION_DIR = str(DATA_ROOT / '6_Province_level_analysis/1_renewable_generation/data/china_solar_wind_complementarity/outputs/hourly_generation')
LOAD_DIR = str(DATA_ROOT / '6_Province_level_analysis/2_province_hourly_load/hourly_load')



PROVINCE_CN_TO_EN = {
    '\u5317\u4eac': 'beijing',
    '\u5929\u6d25': 'tianjin',
    '\u6cb3\u5317': 'hebei',
    '\u5c71\u897f': 'shanxi',
    '\u5c71\u4e1c': 'shandong',
    '\u8499\u897f': 'mengxi',
    '\u8499\u4e1c': 'mengdong',
    '\u5185\u8499': 'neimeng',
    '\u8fbd\u5b81': 'liaoning',
    '\u5409\u6797': 'jilin',
    '\u9ed1\u9f99\u6c5f': 'heilongjiang',
    '\u4e0a\u6d77': 'shanghai',
    '\u6c5f\u82cf': 'jiangsu',
    '\u6d59\u6c5f': 'zhejiang',
    '\u5b89\u5fbd': 'anhui',
    '\u798f\u5efa': 'fujian',
    '\u6c5f\u897f': 'jiangxi',
    '\u6cb3\u5357': 'henan',
    '\u6e56\u5317': 'hubei',
    '\u6e56\u5357': 'hunan',
    '\u91cd\u5e86': 'chongqing',
    '\u56db\u5ddd': 'sichuan',
    '\u9655\u897f': 'shannxi',
    '\u7518\u8083': 'gansu',
    '\u9752\u6d77': 'qinghai',
    '\u5b81\u590f': 'ningxia',
    '\u65b0\u7586': 'xinjiang',
    '\u897f\u85cf': 'xizang',
    '\u5e7f\u4e1c': 'guangdong',
    '\u5e7f\u897f': 'guangxi',
    '\u6d77\u5357': 'hainan',
    '\u8d35\u5dde': 'guizhou',
    '\u4e91\u5357': 'yunnan',
}

def ensure_dir(path):
    'Create the directory if needed.'
    if not os.path.exists(path):
        os.makedirs(path)
    return path


def calculate_curtailment(generation, load, k):
    'Calculate curtailed generation and its share of total generation.'
    excess_power = np.maximum(0, generation - k * load)
    total_excess = np.sum(excess_power)
    total_generation = np.sum(generation)
    curtailment_rate = total_excess / total_generation if total_generation > 0 else 0
    return total_excess, curtailment_rate


def solve_k_by_utilization(generation, load, utilization):
    'Solve for the load scaling factor k from the reported utilization rate.'
    if utilization >= 1.0:
        return None, 0.0, 0.0

    target_rate = 1 - utilization

    def objective(k):
        _, rate = calculate_curtailment(generation, load, k)
        return rate - target_rate

    
    try:
        k_solution = brentq(objective, 0.001, 3.0)
        _, calculated_rate = calculate_curtailment(generation, load, k_solution)
        return k_solution, calculated_rate, target_rate
    except ValueError:
        return None, 0.0, target_rate


def plot_curtailment(generation, load, k, curtailment, year, province_en,
                     province_cn, gen_type, output_dir):
    'Plot generation, the scaled load threshold and curtailed generation.'
    if k is None:
        return None

    fig, ax = plt.subplots(figsize=(16, 6))

    hours = np.arange(len(generation))
    threshold = load * k

    
    ax.fill_between(hours, 0, threshold, alpha=0.2, color='blue', label=f'k x Load (k={k:.4f})')
    ax.plot(threshold, color='blue', linewidth=0.8, alpha=0.8)

    
    ax.fill_between(hours, threshold, generation,
                    where=(generation > threshold),
                    alpha=0.5, color='orange',
                    label=f'Curtailment = {curtailment/1000:.1f} GWh')

    
    type_color = 'orange' if gen_type == 'solar' else 'cyan'
    ax.plot(generation, color=type_color, linewidth=0.5, alpha=0.8,
            label=f'{gen_type.capitalize()} Generation')

    type_name = 'Solar' if gen_type == 'solar' else 'Wind'
    ax.set_xlabel('Hour of Year (2022)', fontsize=11)
    ax.set_ylabel('Power (MW)', fontsize=11)
    ax.set_title(f'{province_en} - {year} {type_name} Curtailment Analysis', fontsize=12)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, len(hours))
    ax.set_ylim(0, max(generation.max(), threshold.max()) * 1.1)

    plt.tight_layout()

    filename = f'{province_en}_{year}_{gen_type}_curtailment.png'
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=150)
    plt.close()

    return filepath


def process_province(province_cn, province_en, year, utilization_solar, utilization_wind,
                     generation_dir, load_dir, output_dir):
    'Process one province and one year.'

    
    solar_gen_df = pd.read_excel(
        os.path.join(generation_dir, 'province_hourly_solar_generation.xlsx'),
        index_col=0
    )
    wind_gen_df = pd.read_excel(
        os.path.join(generation_dir, 'province_hourly_wind_generation.xlsx'),
        index_col=0
    )

    
    if province_en not in solar_gen_df.columns:
        print(f"  [SKIP] {province_cn}: Province is absent from the generation data")
        return None

    solar_power = solar_gen_df[province_en].values * 1000  # GW -> MW
    wind_power = wind_gen_df[province_en].values * 1000    # GW -> MW

    
    load_file = os.path.join(load_dir, f'{province_en}_hourly_load_2019.xlsx')
    if not os.path.exists(load_file):
        print(f"  [SKIP] {province_cn}: Load file not found ({load_file})")
        return None

    load_df = pd.read_excel(load_file)
    hourly_load = pd.to_numeric(load_df['Hourly Load (MW)'], errors='coerce').values

    
    min_len = min(len(solar_power), len(wind_power), len(hourly_load))
    solar_power = solar_power[:min_len]
    wind_power = wind_power[:min_len]
    hourly_load = hourly_load[:min_len]

    print(f"  {province_cn}: {len(hourly_load)} hours, Solar util={utilization_solar:.3f}, Wind util={utilization_wind:.3f}")

    results = {'province_cn': province_cn, 'province_en': province_en, 'year': year}
    results['solar_power'] = solar_power
    results['wind_power'] = wind_power
    results['hourly_load'] = hourly_load

    
    if utilization_solar < 1.0:
        k_solar, solar_rate, target_solar_rate = solve_k_by_utilization(
            solar_power, hourly_load, utilization_solar
        )
        if k_solar is not None:
            curtailment_solar_hourly = np.maximum(0, solar_power - k_solar * hourly_load)
            total_solar_curt = np.sum(curtailment_solar_hourly)

            results['k_solar'] = k_solar
            results['curtailment_solar_MWh'] = total_solar_curt
            results['curtailment_solar_rate'] = solar_rate
            results['curtailment_solar_hourly'] = curtailment_solar_hourly

            
            plot_path = plot_curtailment(
                solar_power, hourly_load, k_solar, total_solar_curt,
                year, province_en, province_cn, 'solar', output_dir
            )
            print(f"    Solar: k={k_solar:.6f}, curtailment={total_solar_curt/1000:.2f} GWh ({solar_rate*100:.2f}%)")
    else:
        results['k_solar'] = None
        results['curtailment_solar_MWh'] = 0
        print(f"    Solar: 100% utilization, skip")

    
    if utilization_wind < 1.0:
        k_wind, wind_rate, target_wind_rate = solve_k_by_utilization(
            wind_power, hourly_load, utilization_wind
        )
        if k_wind is not None:
            curtailment_wind_hourly = np.maximum(0, wind_power - k_wind * hourly_load)
            total_wind_curt = np.sum(curtailment_wind_hourly)

            results['k_wind'] = k_wind
            results['curtailment_wind_MWh'] = total_wind_curt
            results['curtailment_wind_rate'] = wind_rate
            results['curtailment_wind_hourly'] = curtailment_wind_hourly

            
            plot_path = plot_curtailment(
                wind_power, hourly_load, k_wind, total_wind_curt,
                year, province_en, province_cn, 'wind', output_dir
            )
            print(f"    Wind: k={k_wind:.6f}, curtailment={total_wind_curt/1000:.2f} GWh ({wind_rate*100:.2f}%)")
    else:
        results['k_wind'] = None
        results['curtailment_wind_MWh'] = 0
        print(f"    Wind: 100% utilization, skip")

    return results


def save_hourly_curtailment(results, output_dir):
    'Write hourly curtailment to Excel.'
    if results is None:
        return

    province_en = results['province_en']
    province_cn = results['province_cn']
    year = results['year']

    time_index = pd.date_range('2022-01-01', periods=8760, freq='h')

    
    if results.get('k_solar') is not None:
        df_solar = pd.DataFrame({
            'datetime': time_index,
            'solar_generation_MW': results.get('solar_power', []),
            'hourly_load_MW': results.get('hourly_load', []),
            'solar_threshold_MW': results.get('k_solar') * results.get('hourly_load', []),
            'solar_curtailment_MWh': results['curtailment_solar_hourly'],
        })
        solar_file = os.path.join(output_dir, f'{province_en}_{year}_solar_hourly_curtailment.xlsx')
        df_solar.to_excel(solar_file, index=False, engine='openpyxl')
        print(f"    [SAVED] {solar_file}")

    
    if results.get('k_wind') is not None:
        df_wind = pd.DataFrame({
            'datetime': time_index,
            'wind_generation_MW': results.get('wind_power', []),
            'hourly_load_MW': results.get('hourly_load', []),
            'wind_threshold_MW': results.get('k_wind') * results.get('hourly_load', []),
            'wind_curtailment_MWh': results['curtailment_wind_hourly'],
        })
        wind_file = os.path.join(output_dir, f'{province_en}_{year}_wind_hourly_curtailment.xlsx')
        df_wind.to_excel(wind_file, index=False, engine='openpyxl')
        print(f"    [SAVED] {wind_file}")


def main():
    ensure_dir(OUTPUT_DIR)

    years = ['2023', '2024', '2025']

    for year in years:
        print(f"\n{'='*60}")
        print(f" {year}Year: ")
        print('='*60)

        
        provinces_df = pd.read_excel(
            os.path.join(BASE_DIR, 'provinces_list.xlsx'),
            sheet_name=year
        )

        
        all_results = []

        for idx, row in provinces_df.iterrows():
            province_cn = row['\u7701\u4efd/\u5730\u533a']

            
            if province_cn in ['\u8499\u897f', '\u8499\u4e1c']:
                print(f"[SKIP] {province_cn} ")
                continue

            
            if province_cn not in PROVINCE_CN_TO_EN:
                print(f"[SKIP] {province_cn}: Province not found in the province mapping")
                continue

            province_en = PROVINCE_CN_TO_EN[province_cn]
            utilization_solar = row['\u5149\u80fd\u5229\u7528\u7387']
            utilization_wind = row['\u98ce\u80fd\u5229\u7528\u7387']

            print(f"\nProcessing province: {province_cn} ({province_en})")

            
            results = process_province(
                province_cn, province_en, year,
                utilization_solar, utilization_wind,
                GENERATION_DIR, LOAD_DIR, OUTPUT_DIR
            )

            if results:
                
                save_hourly_curtailment(results, OUTPUT_DIR)
                all_results.append(results)

        
        if all_results:
            summary_data = []
            for r in all_results:
                summary_data.append({
                    '\u7701\u4efd_\u4e2d\u6587': r['province_cn'],
                    '\u7701\u4efd_\u82f1\u6587': r['province_en'],
                    '\u5e74\u4efd': r['year'],
                    'k_solar': r.get('k_solar'),
                    'k_wind': r.get('k_wind'),
                    '\u5f03\u5149\u7535\u91cf_MWh': r.get('curtailment_solar_MWh', 0),
                    '\u5f03\u98ce\u7535\u91cf_MWh': r.get('curtailment_wind_MWh', 0),
                    '\u5f03\u5149\u7387': r.get('curtailment_solar_rate', 0),
                    '\u5f03\u98ce\u7387': r.get('curtailment_wind_rate', 0),
                })

            summary_df = pd.DataFrame(summary_data)
            summary_file = os.path.join(OUTPUT_DIR, f'summary_{year}.xlsx')
            summary_df.to_excel(summary_file, index=False, engine='openpyxl')
            print(f"\n[SAVED] {summary_file}")

    print(f"\n{'='*60}")
    print('See the preceding output for processing details.')
    print(f"Output directory: {OUTPUT_DIR}")
    print('='*60)


if __name__ == '__main__':
    main()

# 2.1. Validate modeled curtailment rates

In [ ]:
'Compare modeled curtailment rates with reported rates.'

import pandas as pd
import numpy as np
import os


BASE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k')
HOURLY_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed')
GENERATION_DIR = str(DATA_ROOT / '6_Province_level_analysis/1_renewable_generation/data/china_solar_wind_complementarity/outputs/hourly_generation')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed/check')


PROVINCE_EN_TO_CN = {
    'beijing': '\u5317\u4eac', 'tianjin': '\u5929\u6d25', 'hebei': '\u6cb3\u5317', 'shanxi': '\u5c71\u897f',
    'shandong': '\u5c71\u4e1c', 'liaoning': '\u8fbd\u5b81', 'jilin': '\u5409\u6797', 'heilongjiang': '\u9ed1\u9f99\u6c5f',
    'shanghai': '\u4e0a\u6d77', 'jiangsu': '\u6c5f\u82cf', 'zhejiang': '\u6d59\u6c5f', 'anhui': '\u5b89\u5fbd',
    'fujian': '\u798f\u5efa', 'jiangxi': '\u6c5f\u897f', 'henan': '\u6cb3\u5357', 'hubei': '\u6e56\u5317',
    'hunan': '\u6e56\u5357', 'chongqing': '\u91cd\u5e86', 'sichuan': '\u56db\u5ddd', 'shannxi': '\u9655\u897f',
    'gansu': '\u7518\u8083', 'qinghai': '\u9752\u6d77', 'ningxia': '\u5b81\u590f', 'xinjiang': '\u65b0\u7586',
    'xizang': '\u897f\u85cf', 'guangdong': '\u5e7f\u4e1c', 'guangxi': '\u5e7f\u897f', 'hainan': '\u6d77\u5357',
    'guizhou': '\u8d35\u5dde', 'yunnan': '\u4e91\u5357', 'neimeng': '\u5185\u8499'
}


def main():
    provinces = list(PROVINCE_EN_TO_CN.keys())
    years = ['2023', '2024', '2025']

    print('='*80)
    print('Curtailment-rate validation')
    print('='*80)

    
    solar_gen_df = pd.read_excel(
        os.path.join(GENERATION_DIR, 'province_hourly_solar_generation.xlsx'),
        index_col=0
    )
    wind_gen_df = pd.read_excel(
        os.path.join(GENERATION_DIR, 'province_hourly_wind_generation.xlsx'),
        index_col=0
    )

    
    solar_total_gen = (solar_gen_df.sum() * 1000).to_dict()
    wind_total_gen = (wind_gen_df.sum() * 1000).to_dict()

    all_results = []

    for year in years:
        print(f'\n--- {year}Year: ')

        
        util_df = pd.read_excel(
            os.path.join(BASE_DIR, 'provinces_list.xlsx'),
            sheet_name=year
        )

        for prov in provinces:
            
            if prov not in solar_gen_df.columns:
                continue

            prov_cn = PROVINCE_EN_TO_CN.get(prov, prov)

            
            util_row = util_df[util_df['\u7701\u4efd/\u5730\u533a'] == prov_cn]
            if len(util_row) == 0:
                continue
            util_row = util_row.iloc[0]

            orig_util_solar = util_row['\u5149\u80fd\u5229\u7528\u7387']
            orig_util_wind = util_row['\u98ce\u80fd\u5229\u7528\u7387']
            orig_curt_solar = 1 - orig_util_solar
            orig_curt_wind = 1 - orig_util_wind

            
            solar_file = os.path.join(HOURLY_DIR, f'{prov}_{year}_solar_hourly_curtailment.xlsx')
            wind_file = os.path.join(HOURLY_DIR, f'{prov}_{year}_wind_hourly_curtailment.xlsx')

            solar_curt_rate = None
            wind_curt_rate = None

            
            if os.path.exists(solar_file):
                df_solar = pd.read_excel(solar_file)
                total_solar_curt = df_solar['solar_curtailment_MWh'].sum()
                total_solar_gen = solar_total_gen.get(prov, 0)
                if total_solar_gen > 0:
                    solar_curt_rate = total_solar_curt / total_solar_gen

            
            if os.path.exists(wind_file):
                df_wind = pd.read_excel(wind_file)
                total_wind_curt = df_wind['wind_curtailment_MWh'].sum()
                total_wind_gen = wind_total_gen.get(prov, 0)
                if total_wind_gen > 0:
                    wind_curt_rate = total_wind_curt / total_wind_gen

            
            solar_diff = abs(solar_curt_rate - orig_curt_solar) if solar_curt_rate is not None else None
            wind_diff = abs(wind_curt_rate - orig_curt_wind) if wind_curt_rate is not None else None

            all_results.append({
                '\u7701\u4efd_\u4e2d\u6587': prov_cn,
                '\u7701\u4efd_\u82f1\u6587': prov,
                '\u5e74\u4efd': year,
                '\u539f\u59cb\u5229\u7528\u7387_\u5149': orig_util_solar,
                '\u539f\u59cb\u5f03\u5149\u7387': orig_curt_solar,
                '\u8ba1\u7b97\u5f03\u5149\u7387': solar_curt_rate,
                '\u5f03\u5149\u7387\u5dee\u5f02': solar_diff,
                '\u539f\u59cb\u5229\u7528\u7387_\u98ce': orig_util_wind,
                '\u539f\u59cb\u5f03\u98ce\u7387': orig_curt_wind,
                '\u8ba1\u7b97\u5f03\u98ce\u7387': wind_curt_rate,
                '\u5f03\u98ce\u7387\u5dee\u5f02': wind_diff,
            })

    
    results_df = pd.DataFrame(all_results)

    
    output_file = os.path.join(OUTPUT_DIR, 'curtailment_rate_comparison.xlsx')
    results_df.to_excel(output_file, index=False, engine='openpyxl')
    print(f'\n[SAVED] {output_file}')

    
    print('=== Summary ===')
    valid_solar = results_df[results_df['\u5f03\u5149\u7387\u5dee\u5f02'].notna()]
    valid_wind = results_df[results_df['\u5f03\u98ce\u7387\u5dee\u5f02'].notna()]

    print(f'Solar records: {len(valid_solar)}')
    print(f'Wind records: {len(valid_wind)}')

    if len(valid_solar) > 0:
        print(f'Maximum solar deviation: {valid_solar['\u5f03\u5149\u7387\u5dee\u5f02'].max()*100:.6f}%')
        print(f'Mean solar deviation: {valid_solar['\u5f03\u5149\u7387\u5dee\u5f02'].mean()*100:.6f}%')

    if len(valid_wind) > 0:
        print(f'Maximum wind deviation: {valid_wind['\u5f03\u98ce\u7387\u5dee\u5f02'].max()*100:.6f}%')
        print(f'Mean wind deviation: {valid_wind['\u5f03\u98ce\u7387\u5dee\u5f02'].mean()*100:.6f}%')

    
    print('\n' + '='*90)
    print('Curtailment-rate comparison')
    print('='*90)

    for year in years:
        print(f'\n=== {year}Year: ')
        year_df = results_df[results_df['\u5e74\u4efd'] == int(year)]

        
        has_data = year_df['\u8ba1\u7b97\u5f03\u5149\u7387'].notna() | year_df['\u8ba1\u7b97\u5f03\u98ce\u7387'].notna()
        year_df = year_df[has_data]

        if len(year_df) > 0:
            print(f'{'\u7701\u4efd':<8} {'\u539f\u59cb\u5f03\u5149\u7387':>12} {'\u8ba1\u7b97\u5f03\u5149\u7387':>12} {'\u5f03\u5149\u5dee\u5f02':>10} {'\u539f\u59cb\u5f03\u98ce\u7387':>12} {'\u8ba1\u7b97\u5f03\u98ce\u7387':>12} {'\u5f03\u98ce\u5dee\u5f02':>10}')
            print('-'*90)

            for _, row in year_df.iterrows():
                solar_diff = f'{row['\u5f03\u5149\u7387\u5dee\u5f02']*100:.6f}%' if pd.notna(row['\u5f03\u5149\u7387\u5dee\u5f02']) else '-'
                wind_diff = f'{row['\u5f03\u98ce\u7387\u5dee\u5f02']*100:.6f}%' if pd.notna(row['\u5f03\u98ce\u7387\u5dee\u5f02']) else '-'
                orig_solar = f'{row['\u539f\u59cb\u5f03\u5149\u7387']*100:.2f}%' if row['\u539f\u59cb\u5f03\u5149\u7387'] > 0 else '-'
                orig_wind = f'{row['\u539f\u59cb\u5f03\u98ce\u7387']*100:.2f}%' if row['\u539f\u59cb\u5f03\u98ce\u7387'] > 0 else '-'
                calc_solar = f'{row['\u8ba1\u7b97\u5f03\u5149\u7387']*100:.2f}%' if pd.notna(row['\u8ba1\u7b97\u5f03\u5149\u7387']) else '-'
                calc_wind = f'{row['\u8ba1\u7b97\u5f03\u98ce\u7387']*100:.2f}%' if pd.notna(row['\u8ba1\u7b97\u5f03\u98ce\u7387']) else '-'

                print(f'{row['\u7701\u4efd_\u4e2d\u6587']:<8} {orig_solar:>12} {calc_solar:>12} {solar_diff:>10} {orig_wind:>12} {calc_wind:>12} {wind_diff:>10}')

    print('\n' + '='*90)
    print('Province:')
    print('See the preceding output for processing details.')
    print('='*90)


if __name__ == '__main__':
    main()

# 2.2. Combine wind and solar curtailment

In [ ]:
'Combine hourly wind and solar curtailment files.'

import pandas as pd
import numpy as np
import os


BASE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k')
SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')


def extract_province_year(filename):
    'Extract province year.'
    
    name = filename.replace('_hourly_curtailment.xlsx', '')
    parts = name.split('_')

    
    year = None
    for part in parts:
        if part in ['2023', '2024', '2025']:
            year = part
            break

    
    if year and year in parts:
        idx = parts.index(year)
        province = '_'.join(parts[:idx])  
        return province, year

    return None, None


def merge_curtailment_data():
    'Combine wind and solar curtailment'

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    
    solar_files = [f for f in os.listdir(SOURCE_DIR) if f.endswith('_hourly_curtailment.xlsx') and '_solar_' in f]
    wind_files = [f for f in os.listdir(SOURCE_DIR) if f.endswith('_hourly_curtailment.xlsx') and '_wind_' in f]

    print(f'Solar files found: {len(solar_files)}')
    print(f'Wind files found: {len(wind_files)}')

    
    province_year_solar = {}
    province_year_wind = {}

    for f in solar_files:
        prov, year = extract_province_year(f)
        if prov and year:
            province_year_solar[(prov, year)] = f

    for f in wind_files:
        prov, year = extract_province_year(f)
        if prov and year:
            province_year_wind[(prov, year)] = f

    
    all_keys = set(province_year_solar.keys()) | set(province_year_wind.keys())

    print(f'Province-year pairs: {len(all_keys)}')
    print()

    count = 0
    for (prov, year) in sorted(all_keys):
        has_solar = (prov, year) in province_year_solar
        has_wind = (prov, year) in province_year_wind

        
        if has_solar:
            solar_file = os.path.join(SOURCE_DIR, province_year_solar[(prov, year)])
            df_solar = pd.read_excel(solar_file)
        else:
            df_solar = None

        if has_wind:
            wind_file = os.path.join(SOURCE_DIR, province_year_wind[(prov, year)])
            df_wind = pd.read_excel(wind_file)
        else:
            df_wind = None

        
        if df_solar is not None and df_wind is not None:
            
            result_df = pd.DataFrame({
                'datetime': df_solar['datetime'],
                'solar_generation_MW': df_solar['solar_generation_MW'],
                'wind_generation_MW': df_wind['wind_generation_MW'],
                'solar_curtailment_MWh': df_solar['solar_curtailment_MWh'],
                'wind_curtailment_MWh': df_wind['wind_curtailment_MWh'],
                'total_curtailment_MWh': df_solar['solar_curtailment_MWh'] + df_wind['wind_curtailment_MWh'],
            })
            source = 'both'
        elif df_solar is not None:
            
            result_df = pd.DataFrame({
                'datetime': df_solar['datetime'],
                'solar_generation_MW': df_solar['solar_generation_MW'],
                'wind_generation_MW': 0.0,
                'solar_curtailment_MWh': df_solar['solar_curtailment_MWh'],
                'wind_curtailment_MWh': 0.0,
                'total_curtailment_MWh': df_solar['solar_curtailment_MWh'],
            })
            source = 'solar only'
        else:
            
            result_df = pd.DataFrame({
                'datetime': df_wind['datetime'],
                'solar_generation_MW': 0.0,
                'wind_generation_MW': df_wind['wind_generation_MW'],
                'solar_curtailment_MWh': 0.0,
                'wind_curtailment_MWh': df_wind['wind_curtailment_MWh'],
                'total_curtailment_MWh': df_wind['wind_curtailment_MWh'],
            })
            source = 'wind only'

        
        output_file = os.path.join(OUTPUT_DIR, f'{prov}_{year}_total_hourly_curtailment.xlsx')
        result_df.to_excel(output_file, index=False, engine='openpyxl')

        total = result_df['total_curtailment_MWh'].sum()
        print(f'{prov}_{year}: [{source}] Total = {total/1000:.2f} GWh')
        count += 1

    print(f'Wrote {count} files.')
    print(f'Output directory: {OUTPUT_DIR}')


def check_completeness():
    'Check whether all required province-year pairs exist.'
    print('\n' + '='*60)
    print('Data completeness check')
    print('='*60)

    
    CN_TO_EN = {
        '\u5317\u4eac': 'beijing', '\u5929\u6d25': 'tianjin', '\u6cb3\u5317': 'hebei', '\u5c71\u897f': 'shanxi',
        '\u5c71\u4e1c': 'shandong', '\u8fbd\u5b81': 'liaoning', '\u5409\u6797': 'jilin', '\u9ed1\u9f99\u6c5f': 'heilongjiang',
        '\u4e0a\u6d77': 'shanghai', '\u6c5f\u82cf': 'jiangsu', '\u6d59\u6c5f': 'zhejiang', '\u5b89\u5fbd': 'anhui',
        '\u798f\u5efa': 'fujian', '\u6c5f\u897f': 'jiangxi', '\u6cb3\u5357': 'henan', '\u6e56\u5317': 'hubei',
        '\u6e56\u5357': 'hunan', '\u91cd\u5e86': 'chongqing', '\u56db\u5ddd': 'sichuan', '\u9655\u897f': 'shannxi',
        '\u7518\u8083': 'gansu', '\u9752\u6d77': 'qinghai', '\u5b81\u590f': 'ningxia', '\u65b0\u7586': 'xinjiang',
        '\u897f\u85cf': 'xizang', '\u5e7f\u4e1c': 'guangdong', '\u5e7f\u897f': 'guangxi', '\u6d77\u5357': 'hainan',
        '\u8d35\u5dde': 'guizhou', '\u4e91\u5357': 'yunnan', '\u5185\u8499': 'neimeng'
    }

    
    total_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.xlsx')]
    existing = {}
    for f in total_files:
        prov, year = extract_province_year(f.replace('_total_', '_'))
        if prov and year:
            existing[(prov, year)] = f

    
    all_needed = set()
    for year in ['2023', '2024', '2025']:
        df = pd.read_excel(f'{BASE_DIR}/provinces_list.xlsx', sheet_name=year)
        for _, row in df.iterrows():
            prov_cn = row['\u7701\u4efd/\u5730\u533a']
            if prov_cn in ['\u8499\u897f', '\u8499\u4e1c']:
                continue
            prov_en = CN_TO_EN.get(prov_cn)
            if prov_en:
                all_needed.add((prov_en, year))

    missing = all_needed - set(existing.keys())

    print(f'Available records: {len(existing)} ')
    print(f'Required records: {len(all_needed)} ')
    print(f'Missing records: {len(missing)} ')

    if missing:
        print('Missing combinations:')
        for (prov, year) in sorted(missing):
            
            solar_file = os.path.join(SOURCE_DIR, f'{prov}_{year}_solar_hourly_curtailment.xlsx')
            wind_file = os.path.join(SOURCE_DIR, f'{prov}_{year}_wind_hourly_curtailment.xlsx')
            has_solar = os.path.exists(solar_file)
            has_wind = os.path.exists(wind_file)
            reason = ''
            if not has_solar and not has_wind:
                reason = ' (\u539f\u59cb\u6570\u636e\u4e0d\u5b58\u5728)'
            elif not has_solar:
                reason = ' (\u53ea\u6709\u98ce\u7535\u6570\u636e)'
            elif not has_wind:
                reason = ' (\u53ea\u6709\u5149\u4f0f\u6570\u636e)'
            print(f'  {prov}_{year}{reason}')


if __name__ == '__main__':
    print('='*60)
    print('Combine wind and solar curtailment')
    print('='*60)

    merge_curtailment_data()
    check_completeness()


# 2.3. Summarize annual provincial curtailment

In [ ]:
import os
import re
from pathlib import Path
 
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
 

SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
OUTPUT_FILE = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/province_annual_curtailment.xlsx')
 

PROVINCE_ORDER = [
    ("beijing", '\u5317\u4eac'), ("tianjin", '\u5929\u6d25'), ("hebei", '\u6cb3\u5317'), ("shanxi", '\u5c71\u897f'),
    ("neimeng", '\u5185\u8499\u53e4'), ("liaoning", '\u8fbd\u5b81'), ("jilin", '\u5409\u6797'), ("heilongjiang", '\u9ed1\u9f99\u6c5f'),
    ("shanghai", '\u4e0a\u6d77'), ("jiangsu", '\u6c5f\u82cf'), ("zhejiang", '\u6d59\u6c5f'), ("anhui", '\u5b89\u5fbd'),
    ("fujian", '\u798f\u5efa'), ("jiangxi", '\u6c5f\u897f'), ("shandong", '\u5c71\u4e1c'), ("henan", '\u6cb3\u5357'),
    ("hubei", '\u6e56\u5317'), ("hunan", '\u6e56\u5357'), ("guangdong", '\u5e7f\u4e1c'), ("guangxi", '\u5e7f\u897f'),
    ("hainan", '\u6d77\u5357'), ("chongqing", '\u91cd\u5e86'), ("sichuan", '\u56db\u5ddd'), ("guizhou", '\u8d35\u5dde'),
    ("yunnan", '\u4e91\u5357'), ("xizang", '\u897f\u85cf'), ("shannxi", '\u9655\u897f'), ("gansu", '\u7518\u8083'),
    ("qinghai", '\u9752\u6d77'), ("ningxia", '\u5b81\u590f'), ("xinjiang", '\u65b0\u7586'),
]
CN_NAME = dict(PROVINCE_ORDER)
 
YEARS = [2023, 2024, 2025]
 
FILE_SUFFIX = "_total_hourly_curtailment.xlsx"
 
 
def extract_province_year(filename: str):
    'Extract province year.'
    if not filename.endswith(FILE_SUFFIX):
        return None, None
    name = filename[: -len(FILE_SUFFIX)]
    parts = name.split("_")
    for i, p in enumerate(parts):
        if re.fullmatch(r"20\d{2}", p):
            province = "_".join(parts[:i])
            return province, int(p)
    return None, None
 
 
def scan_source_dir(source_dir: str) -> pd.DataFrame:
    'Scan source dir.'
    records = []
    files = [f for f in os.listdir(source_dir) if f.endswith(FILE_SUFFIX)]
    print(f"Found {len(files)} input files in {source_dir}.")
 
    for fname in sorted(files):
        prov, year = extract_province_year(fname)
        if prov is None:
            print(f"  [Skip] Could not parse filename: {fname}")
            continue
 
        fp = os.path.join(source_dir, fname)
        df = pd.read_excel(fp)
 
        if "total_curtailment_MWh" not in df.columns:
            print(f"  [Skip] {fname}: Missing total_curtailment_MWh column")
            continue
 
        total_mwh = df["total_curtailment_MWh"].sum()
        solar_mwh = df["solar_curtailment_MWh"].sum() if "solar_curtailment_MWh" in df.columns else None
        wind_mwh = df["wind_curtailment_MWh"].sum() if "wind_curtailment_MWh" in df.columns else None
 
        if solar_mwh is not None and wind_mwh is not None:
            has_solar = solar_mwh > 0
            has_wind = wind_mwh > 0
            if has_solar and has_wind:
                source = "both"
            elif has_solar:
                source = "solar only"
            elif has_wind:
                source = "wind only"
            else:
                source = "none"
        else:
            source = "unknown"
 
        total_gwh = total_mwh / 1000.0
        records.append(
            {"province": prov, "year": year, "total_curtailment_GWh": total_gwh, "source": source}
        )
        print(f"  {prov}_{year}: [{source}] Total = {total_gwh:.2f} GWh")
 
    return pd.DataFrame(records)
 
 
def build_summary_workbook(data_df: pd.DataFrame, output_file: str):
    'Build summary workbook.'
    data = {
        (row.province, row.year): (row.total_curtailment_GWh, row.source)
        for row in data_df.itertuples()
    }
 
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Annual curtailment by province'
 
    FONT_NAME = "Arial"
    header_font = Font(name=FONT_NAME, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    title_font = Font(name=FONT_NAME, bold=True, size=14)
    normal_font = Font(name=FONT_NAME)
    partial_font = Font(name=FONT_NAME, color="B45F06")  
    bold_font = Font(name=FONT_NAME, bold=True)
    na_font = Font(name=FONT_NAME, color="999999", italic=True)
    thin = Side(style="thin", color="B7B7B7")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", vertical="center")
    right = Alignment(horizontal="right", vertical="center")
 
    ws.merge_cells("A1:F1")
    ws["A1"] = 'Annual provincial wind and solar curtailment (GWh)'
    ws["A1"].font = title_font
    ws["A1"].alignment = center
 
    ws["A2"] = 'Source: hourly province-year files. Hourly curtailment (MWh) is summed and converted to GWh.'
    ws["A2"].font = Font(name=FONT_NAME, italic=True, size=9, color="666666")
    ws.merge_cells("A2:F2")
 
    header_row = 4
    headers = ['Province key', 'Province (source name)'] + [str(y) for y in YEARS] + ['Three-year total (GWh)']
    for col, h in enumerate(headers, start=1):
        c = ws.cell(row=header_row, column=col, value=h)
        c.font = header_font
        c.fill = header_fill
        c.alignment = center
        c.border = border
 
    
    known_provinces = {p for p, _ in PROVINCE_ORDER}
    extra_provinces = sorted({p for p in data_df["province"].unique() if p not in known_provinces})
    all_provinces = PROVINCE_ORDER + [(p, CN_NAME.get(p, "")) for p in extra_provinces]
 
    start_row = header_row + 1
    for i, (prov_en, prov_cn) in enumerate(all_provinces):
        r = start_row + i
        ws.cell(row=r, column=1, value=prov_en).font = normal_font
        ws.cell(row=r, column=2, value=prov_cn).font = normal_font
 
        for j, year in enumerate(YEARS):
            col = 3 + j
            cell = ws.cell(row=r, column=col)
            if (prov_en, year) in data:
                total, source = data[(prov_en, year)]
                cell.value = round(total, 2)
                cell.number_format = "#,##0.00"
                cell.font = normal_font if source == "both" else partial_font
            else:
                cell.value = 'No data'
                cell.font = na_font
            cell.alignment = right
            cell.border = border
 
        ws.cell(row=r, column=1).border = border
        ws.cell(row=r, column=2).border = border
 
        total_cell = ws.cell(row=r, column=6)
        total_cell.value = f"=SUM(C{r}:E{r})"
        total_cell.number_format = "#,##0.00"
        total_cell.font = bold_font
        total_cell.alignment = right
        total_cell.border = border
 
    end_row = start_row + len(all_provinces) - 1
 
    gt_row = end_row + 1
    ws.cell(row=gt_row, column=2, value='National total').font = bold_font
    ws.cell(row=gt_row, column=2).alignment = right
    ws.cell(row=gt_row, column=1).border = border
    ws.cell(row=gt_row, column=2).border = border
    for j, year in enumerate(YEARS):
        col = 3 + j
        letter = get_column_letter(col)
        c = ws.cell(row=gt_row, column=col)
        c.value = f"=SUM({letter}{start_row}:{letter}{end_row})"
        c.number_format = "#,##0.00"
        c.font = bold_font
        c.alignment = right
        c.border = border
    c = ws.cell(row=gt_row, column=6)
    c.value = f"=SUM(F{start_row}:F{end_row})"
    c.number_format = "#,##0.00"
    c.font = bold_font
    c.alignment = right
    c.border = border
 
    note_row = gt_row + 2
    ws.cell(row=note_row, column=1, value='Notes:').font = bold_font
    notes = [
        '1. "No data" means the hourly input file is missing for that province and year.',
        '2. Orange values indicate that only solar or only wind data are available. Totals may be understated.',
        '3. Black values indicate that both solar and wind data are available.',
    ]
    for i, note in enumerate(notes):
        ws.cell(row=note_row + 1 + i, column=1, value=note).font = Font(
            name=FONT_NAME, size=9, color="444444"
        )
        ws.merge_cells(start_row=note_row + 1 + i, start_column=1, end_row=note_row + 1 + i, end_column=6)
 
    widths = [14, 12, 12, 12, 12, 16]
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w
 
    ws.freeze_panes = f"C{start_row}"
 
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_file)
    print(f"\n[OK] Summary saved: {output_file}")
 
 
def main():
    data_df = scan_source_dir(SOURCE_DIR)
    if data_df.empty:
        print('See the preceding output for processing details.')
        return
    build_summary_workbook(data_df, OUTPUT_FILE)
 
    
    pivot = data_df.pivot_table(
        index="province", columns="year", values="total_curtailment_GWh", aggfunc="sum"
    )
    print('See the preceding output for processing details.')
    print(pivot.round(2))
 
 
if __name__ == "__main__":
    main()

# 3. Data center operating scenarios

# 3.1. Constant high load

In [ ]:
'Simulate constant high-load operation at 70% curtailment utilization.'

import os
import numpy as np
import pandas as pd



SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/scenario1_high')

TARGET_RATE = 0.70
VALID_YEARS = ['2022', '2023', '2024', '2025']


def search_k_constant(curt, target_rate=TARGET_RATE):
    'Search k constant.'
    total_curt = curt.sum()
    if total_curt <= 0:
        return 0.0

    k_lo, k_hi = 0.0, curt.max()

    for _ in range(100):
        k_mid = (k_lo + k_hi) / 2
        rate = np.minimum(curt, k_mid).sum() / total_curt

        if abs(rate - target_rate) < 1e-8:
            return k_mid
        if rate < target_rate:
            k_lo = k_mid
        else:
            k_hi = k_mid

    return (k_lo + k_hi) / 2


def parse_province_year(filename):
    'Parse province year.'
    suffix = '_total_hourly_curtailment.xlsx'
    base_name = filename[:-len(suffix)]

    year = None
    for part in base_name.split('_'):
        if part in VALID_YEARS:
            year = int(part)
            break

    if year is None:
        raise ValueError(f'Unable to identify year from filename:{filename}')

    province = base_name.rsplit(f'_{year}', 1)[0]
    return province, year


def generate_data_center_scenario():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    total_files = sorted([
        f for f in os.listdir(SOURCE_DIR)
        if f.endswith('_total_hourly_curtailment.xlsx')
    ])

    print(f'Found {len(total_files)} curtailment files.')

    k_stats = []
    count = 0

    for f in total_files:
        df = pd.read_excel(os.path.join(SOURCE_DIR, f))

        if 'datetime' not in df.columns or 'total_curtailment_MWh' not in df.columns:
            print(f'Skip {f}:Required columns are missing')
            continue

        curt = df['total_curtailment_MWh'].to_numpy(dtype=float)
        curt = np.nan_to_num(curt, nan=0.0, posinf=0.0, neginf=0.0)
        curt = np.maximum(curt, 0.0)

        total_curt = curt.sum()
        province, year = parse_province_year(f)

        
        k = search_k_constant(curt)

        
        dc_load = np.full_like(curt, k, dtype=float)
        greenload = np.minimum(curt, dc_load)
        baseload = np.maximum(0.0, dc_load - curt)
        unused = curt - greenload

        
        dt_series = pd.to_datetime(df['datetime']).apply(lambda x: x.replace(year=year))

        result_df = pd.DataFrame({
            'datetime': dt_series,
            'curtailed_power_MW': curt,
            'data_center_load_MW': dc_load,
            'baseload_MW': baseload,
            'greenload_MW': greenload,
            'unused_curtailed_power_MW': unused
        })

        
        output_filename = f'{province}_{year}_scenario1_high.xlsx'
        output_file = os.path.join(OUTPUT_DIR, output_filename)
        result_df.to_excel(output_file, index=False, engine='openpyxl')

        # Summary
        total_green = greenload.sum()
        total_base = baseload.sum()
        total_unused = unused.sum()
        rate = total_green / total_curt * 100 if total_curt > 0 else 0.0

        print(
            f'{province}_{year}: k={k:.2f} MW, '
            f'Curt={total_curt / 1000:.2f} GWh, '
            f'\u6d88\u7eb3={total_green / 1000:.2f} GWh ({rate:.2f}%), '
            f'\u672a\u6d88\u7eb3={total_unused / 1000:.2f} GWh'
        )

        k_stats.append({
            'province': province,
            'year': year,
            'k_MW': k,
            'total_curt_GWh': total_curt / 1000,
            'total_green_GWh': total_green / 1000,
            'total_baseload_GWh': total_base / 1000,
            'total_unused_GWh': total_unused / 1000,
            'green_rate_%': rate
        })

        count += 1

    
    k_df = pd.DataFrame(k_stats).sort_values(['year', 'province']).reset_index(drop=True)
    summary_file = os.path.join(OUTPUT_DIR, 'scenario1_high_k_summary.xlsx')
    k_df.to_excel(summary_file, index=False, engine='openpyxl')

    print(f'Wrote {count} files.')
    print(f' ')


if __name__ == '__main__':
    print('=' * 60)
    print('Operating scenario:')
    print('=' * 60)
    generate_data_center_scenario()

# 3.2. Demand-following load

In [ ]:
'Simulate demand-following operation using a normalized daily load profile.'

import pandas as pd
import numpy as np
import os



SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
TYPICAL_FILE = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/scenario2_typical/typical/typical_load.csv')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/scenario2_typical')


def extract_province_year(filename):
    'Extract province year.'
    name = filename.replace('_total_hourly_curtailment.xlsx', '')
    parts = name.split('_')
    year = None
    for p in parts:
        if p in ['2023', '2024', '2025']:
            year = int(p)
            break
    if not year:
        return None, None
    province = '_'.join(parts[:parts.index(str(year))])
    return province, year


def load_typical_day_pattern():
    'Load typical day pattern.'
    df = pd.read_csv(TYPICAL_FILE, encoding='utf-8')
    raw = df['request'].values
    return raw / raw.max()


def build_lf_sequence(curtailment_df):
    'Build lf sequence.'
    dt = pd.to_datetime(curtailment_df['datetime'])
    year = dt.dt.year.iloc[0]

    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

    hour_of_year = dt.dt.dayofyear - 1
    hour_idx = dt.dt.hour

    if is_leap:
        leap_mask = hour_of_year >= 60
        hour_of_year = hour_of_year - leap_mask.astype(int)

    hour_of_day = (hour_of_year * 24 + hour_idx) % 24

    typical_pattern = load_typical_day_pattern()
    lf_seq = typical_pattern[hour_of_day.values]
    return lf_seq


def search_k(curt, lf, target_rate=0.7):
    'Search k.'
    total_curt = curt.sum()

    k_lo, k_hi = 0.0, curt.max() * 2

    for _ in range(100):
        k_mid = (k_lo + k_hi) / 2
        dc_load = k_mid * lf
        greenload = np.minimum(curt, dc_load)
        rate = greenload.sum() / total_curt

        if abs(rate - target_rate) < 1e-8:
            break
        if rate < target_rate:
            k_lo = k_mid
        else:
            k_hi = k_mid

    return (k_lo + k_hi) / 2


def process_all_provinces():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    typical_pattern = load_typical_day_pattern()
    print('Typical daily load factor (normalized, 70% utilization)')
    print(f' {typical_pattern.min():.4f}, {typical_pattern.max():.4f}]')
    print()

    total_files = sorted([
        f for f in os.listdir(SOURCE_DIR)
        if f.endswith('_total_hourly_curtailment.xlsx')
    ])
    print(f'Found {len(total_files)} curtailment files.')

    k_stats = []

    for fname in total_files:
        province, year = extract_province_year(fname)
        if province is None or year is None:
            print(f' {fname}')
            continue

        df = pd.read_excel(os.path.join(SOURCE_DIR, fname))
        curt = df['total_curtailment_MWh'].values
        total_curt = curt.sum()

        lf = build_lf_sequence(df)

        
        k = search_k(curt, lf, target_rate=0.7)

        dc_load = k * lf
        greenload = np.minimum(curt, dc_load)
        baseload = np.maximum(0.0, dc_load - curt)
        unused = curt - greenload

        dt_series = pd.to_datetime(df['datetime']).apply(lambda x: x.replace(year=year))

        result_df = pd.DataFrame({
            'datetime': dt_series,
            'curtailed_power_MW': curt,
            'data_center_load_MW': dc_load,
            'baseload_MW': baseload,
            'greenload_MW': greenload,
            'unused_curtailed_power_MW': unused,
        })

        output_fname = f'{province}_{year}_scenario2_typical.xlsx'
        result_df.to_excel(os.path.join(OUTPUT_DIR, output_fname), index=False, engine='openpyxl')

        total_green = greenload.sum()
        total_base = baseload.sum()
        rate = total_green / total_curt * 100

        print(f'  {province}_{year}: k={k:.1f} MW, Curt={total_curt/1000:.1f} GWh, '
              f'\u6d88\u7eb3={total_green/1000:.1f} GWh ({rate:.2f}%), '
              f'\u672a\u6d88\u7eb3={unused.sum()/1000:.1f} GWh')

        k_stats.append({
            'province': province,
            'year': year,
            'k_MW': k,
            'total_curt_GWh': total_curt / 1000,
            'total_green_GWh': total_green / 1000,
            'total_baseload_GWh': total_base / 1000,
            'total_unused_GWh': unused.sum() / 1000,
            'green_rate_%': rate,
        })

    k_df = pd.DataFrame(k_stats).sort_values(['year', 'province']).reset_index(drop=True)
    k_summary_path = os.path.join(OUTPUT_DIR, 'scenario2_70_k_summary.xlsx')
    k_df.to_excel(k_summary_path, index=False, engine='openpyxl')

    print(f'Processed {len(k_stats)} files.')
    print(f'Summary saved: {k_summary_path}')

    return k_df


if __name__ == '__main__':
    print('=' * 60)
    print('Operating scenario:')
    print('=' * 60)
    process_all_provinces()

# 3.3. Curtailment-following load

In [ ]:
'Simulate curtailment-following operation within the configured power bounds.'

import pandas as pd
import numpy as np
import os


SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/scenario3_flexible')


def extract_province_year(filename):
    'Extract province year.'
    name = filename.replace('_total_hourly_curtailment.xlsx', '')
    parts = name.split('_')
    year = None
    for p in parts:
        if p in ['2023', '2024', '2025']:
            year = int(p)
            break
    if not year:
        return None, None
    province = '_'.join(parts[:parts.index(str(year))])
    return province, year


def search_k_piecewise(curt, target_rate=0.7):
    'Search k piecewise.'
    total_curt = curt.sum()
    k_lo, k_hi = 0.0, curt.max()

    for _ in range(100):
        k_mid = (k_lo + k_hi) / 2
        p_min = 0.23 * k_mid   
        p_max = 0.88 * k_mid   
        dc_load = np.maximum(p_min, np.minimum(curt, p_max))
        greenload = np.minimum(curt, dc_load)
        rate = greenload.sum() / total_curt

        if abs(rate - target_rate) < 1e-8:
            break
        if rate < target_rate:
            k_lo = k_mid
        else:
            k_hi = k_mid

    return (k_lo + k_hi) / 2


def process_all_provinces():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    total_files = sorted([
        f for f in os.listdir(SOURCE_DIR)
        if f.endswith('_total_hourly_curtailment.xlsx')
    ])
    print(f'Found {len(total_files)} curtailment files.')
    print()

    k_stats = []
    failed = []

    for fname in total_files:
        province, year = extract_province_year(fname)
        if province is None:
            failed.append(fname)
            continue

        df = pd.read_excel(os.path.join(SOURCE_DIR, fname))
        curt = df['total_curtailment_MWh'].values
        total_curt = curt.sum()

        
        k = search_k_piecewise(curt, target_rate=0.7)

        p_min = 0.23 * k
        p_max = 0.88 * k

        dc_load = np.maximum(p_min, np.minimum(curt, p_max))
        greenload = np.minimum(curt, dc_load)
        baseload = np.maximum(0.0, dc_load - curt)
        unused = curt - greenload

        dt_series = pd.to_datetime(df['datetime']).apply(lambda x: x.replace(year=year))

        result_df = pd.DataFrame({
            'datetime': dt_series,
            'curtailed_power_MW': curt,
            'data_center_load_MW': dc_load,
            'baseload_MW': baseload,
            'greenload_MW': greenload,
            'unused_curtailed_power_MW': unused,
        })

        output_fname = f'{province}_{year}_scenario3_flexible.xlsx'
        result_df.to_excel(os.path.join(OUTPUT_DIR, output_fname), index=False, engine='openpyxl')

        total_green = greenload.sum()
        total_base = baseload.sum()
        rate = total_green / total_curt * 100

        print(f'  {province}_{year}: k={k:.1f} MW, P_max={p_max:.1f} MW, P_min={p_min:.1f} MW, '
              f'Curt={total_curt/1000:.1f} GWh, '
              f'\u6d88\u7eb3={total_green/1000:.1f} GWh ({rate:.2f}%), '
              f'\u672a\u6d88\u7eb3={unused.sum()/1000:.1f} GWh')

        k_stats.append({
            'province': province,
            'year': year,
            'k_MW': k,
            'P_max_MW': p_max,
            'P_min_MW': p_min,
            'total_curt_GWh': total_curt / 1000,
            'total_green_GWh': total_green / 1000,
            'total_baseload_GWh': total_base / 1000,
            'total_unused_GWh': unused.sum() / 1000,
            'green_rate_%': rate,
        })

    k_df = pd.DataFrame(k_stats)
    k_df = k_df.sort_values(['year', 'province']).reset_index(drop=True)
    k_summary_path = os.path.join(OUTPUT_DIR, 'scenario3new_70_k_summary.xlsx')
    k_df.to_excel(k_summary_path, index=False, engine='openpyxl')

    print(f'Processed {len(k_stats)} files.')
    if failed:
        print(f'Skip: {failed}')
    print(f'Summary saved: {k_summary_path}')

    return k_df


if __name__ == '__main__':
    print('=' * 60)
    print('Operating scenario:')
    print('Power range: [0.23k, 0.88k]')
    print('=' * 60)
    process_all_provinces()

# 4. Estimate installed capacity

In [ ]:
'Estimate rated capacity from the peak operating load in each scenario.'
 
import pandas as pd
import os
 

BASE_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter')
 

SCENARIOS = [
    ('scenario1_high', '_scenario1_high.xlsx', 0.75),
    ('scenario2_typical', '_scenario2_typical.xlsx', 0.88),
    ('scenario3_flexible', '_scenario3_flexible.xlsx', 0.88),
]
 
OUTPUT_FILE = os.path.join(BASE_DIR, 'capacity','province_annual_installed_capacity.xlsx')
 
 
def extract_province_year(filename, suffix):
    'Extract province year.'
    if not filename.endswith(suffix):
        return None, None
    name = filename.replace(suffix, '')
    parts = name.split('_')
    year = None
    for p in parts:
        if p in ['2023', '2024', '2025']:
            year = int(p)
            break
    if year is None:
        return None, None
    province = '_'.join(parts[:parts.index(str(year))])
    return province, year
 
 

records = []
 
for scenario_folder, suffix, utilization_cap in SCENARIOS:
    folder_path = os.path.join(BASE_DIR, scenario_folder)
    files = [f for f in os.listdir(folder_path) if f.endswith(suffix)]
 
    print(f'{scenario_folder}: {len(files)} files, capacity utilization cap {utilization_cap}.')
 
    for fname in sorted(files):
        province, year = extract_province_year(fname, suffix)
        if province is None:
            print(f'Skipping {fname}: no matching province.')
            continue
 
        df = pd.read_excel(os.path.join(folder_path, fname))
        max_load = df['data_center_load_MW'].max()
        installed_capacity_MW = max_load / utilization_cap
 
        records.append({
            'province': province,
            'year': year,
            'scenario': scenario_folder,
            'installed_capacity_MW': installed_capacity_MW,
        })
 

long_df = pd.DataFrame(records)
 
wide_df = long_df.pivot_table(
    index=['province', 'year'],
    columns='scenario',
    values='installed_capacity_MW'
).reset_index()
 

scenario_cols = [s for s, _, _ in SCENARIOS]
wide_df = wide_df[['province', 'year'] + scenario_cols]
wide_df.columns = ['province', 'year'] + [f'{s}_installed_capacity_MW' for s in scenario_cols]
 
wide_df = wide_df.sort_values(['province', 'year']).reset_index(drop=True)
 
wide_df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
 
print(f'Wrote {len(wide_df)} province-year records.')
print(f'Output file: {OUTPUT_FILE}')

# 5. Sweep curtailment utilization targets from 10% to 100%

In [ ]:



import os
import numpy as np
import pandas as pd

# ============ Paths ============
SOURCE_DIR = str(DATA_ROOT / '6_Province_level_analysis/3_curtailment_k/hourly_curtailed_solar_wind_merge')
TYPICAL_FILE = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/scenario2_typical/typical/typical_load.csv')
OUTPUT_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter/gradient')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'gradient_scenarios_summary.xlsx')

# ============ Settings ============
UTIL_TARGETS = np.round(np.arange(0.1, 1.001, 0.1), 2)      # 10%, 20%, ..., 100%
P_MIN_FRAC, P_MAX_FRAC = 0.23, 0.88                           # scenario3 power band [0.23k, 0.88k]
CAPACITY_CAP = {                                              # rated-utilization cap, same as Section 4
    'scenario1_high': 0.75,
    'scenario2_typical': 0.88,
    'scenario3_flexible': 0.88,
}
VALID_YEARS = ['2023', '2024', '2025']

os.makedirs(OUTPUT_DIR, exist_ok=True)

# typical-day load factor (normalised to max = 1)
typical = pd.read_csv(TYPICAL_FILE, encoding='utf-8')['request'].to_numpy(dtype=float)
typical = typical / typical.max()

files = sorted(f for f in os.listdir(SOURCE_DIR) if f.endswith('_total_hourly_curtailment.xlsx'))
print(f'Found {len(files)} curtailment files')

records = []

for fname in files:
    parts = fname.replace('_total_hourly_curtailment.xlsx', '').split('_')
    year_str = next((p for p in parts if p in VALID_YEARS), None)
    if year_str is None:
        print(f'  [skip] cannot parse year: {fname}')
        continue
    province = '_'.join(parts[:parts.index(year_str)])
    year = int(year_str)

    df = pd.read_excel(os.path.join(SOURCE_DIR, fname))
    curt = np.nan_to_num(df['total_curtailment_MWh'].to_numpy(dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    curt = np.maximum(curt, 0.0)
    total_curt = curt.sum()
    lf = typical[pd.to_datetime(df['datetime']).dt.hour.to_numpy()]

    for scenario in CAPACITY_CAP:
        for target in UTIL_TARGETS:

            if total_curt <= 0:
                records.append({
                    'province': province, 'year': year, 'scenario': scenario,
                    'target_utilization_%': round(target * 100),
                    'curtailment_utilization_%': np.nan, 'dc_renewable_share_%': np.nan,
                    'installed_capacity_MW': 0.0, 'k_MW': 0.0,
                    'curt_GWh': 0.0, 'green_GWh': 0.0, 'dc_energy_GWh': 0.0,
                })
                continue

            # upper bound of k so that 100% utilization is reachable in every scenario
            k_lo = 0.0
            if scenario == 'scenario1_high':
                k_hi = curt.max()
            elif scenario == 'scenario2_typical':
                k_hi = curt.max() / max(lf.min(), 1e-6)
            else:
                k_hi = curt.max() / P_MAX_FRAC

            # bisection on k: sum(min(curt, dc_load)) / sum(curt) = target
            for _ in range(100):
                k = (k_lo + k_hi) / 2
                if scenario == 'scenario1_high':
                    dc_load = np.full_like(curt, k)
                elif scenario == 'scenario2_typical':
                    dc_load = k * lf
                else:
                    dc_load = np.maximum(P_MIN_FRAC * k, np.minimum(curt, P_MAX_FRAC * k))
                rate = np.minimum(curt, dc_load).sum() / total_curt
                if abs(rate - target) < 1e-8:
                    break
                if rate < target:
                    k_lo = k
                else:
                    k_hi = k

            greenload = np.minimum(curt, dc_load)

            records.append({
                'province': province,
                'year': year,
                'scenario': scenario,
                'target_utilization_%': round(target * 100),
                'curtailment_utilization_%': greenload.sum() / total_curt * 100,
                'dc_renewable_share_%': greenload.sum() / dc_load.sum() * 100,
                'installed_capacity_MW': dc_load.max() / CAPACITY_CAP[scenario],
                'k_MW': k,
                'curt_GWh': total_curt / 1000,
                'green_GWh': greenload.sum() / 1000,
                'dc_energy_GWh': dc_load.sum() / 1000,
            })

    print(f'  done: {province}_{year}')

# ============ Province-year table ============
scenario_order = {s: i for i, s in enumerate(CAPACITY_CAP)}
prov_df = pd.DataFrame(records)
prov_df['_s'] = prov_df['scenario'].map(scenario_order)
prov_df = (prov_df.sort_values(['_s', 'year', 'province', 'target_utilization_%'])
                  .drop(columns='_s').reset_index(drop=True))

# ============ National table (sum over provinces) ============
nat_df = (prov_df.groupby(['scenario', 'year', 'target_utilization_%'], as_index=False)
                 .agg(curt_GWh=('curt_GWh', 'sum'),
                      green_GWh=('green_GWh', 'sum'),
                      dc_energy_GWh=('dc_energy_GWh', 'sum'),
                      installed_capacity_MW=('installed_capacity_MW', 'sum')))
nat_df['curtailment_utilization_%'] = nat_df['green_GWh'] / nat_df['curt_GWh'] * 100
nat_df['dc_renewable_share_%'] = nat_df['green_GWh'] / nat_df['dc_energy_GWh'] * 100
nat_df['_s'] = nat_df['scenario'].map(scenario_order)
nat_df = (nat_df.sort_values(['_s', 'year', 'target_utilization_%']).drop(columns='_s')
                [['scenario', 'year', 'target_utilization_%', 'curtailment_utilization_%',
                  'dc_renewable_share_%', 'installed_capacity_MW',
                  'curt_GWh', 'green_GWh', 'dc_energy_GWh']]
                .reset_index(drop=True))

# ============ Save ============
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    prov_df.to_excel(writer, sheet_name='province_year', index=False)
    nat_df.to_excel(writer, sheet_name='national', index=False)

print(f'\nSaved: {OUTPUT_FILE}')
print(f'province_year rows: {len(prov_df)}, national rows: {len(nat_df)}')

# 6. Estimate carbon emissions by operating scenario

In [ ]:
'Estimate grid-related emissions for each operating scenario.'

import pandas as pd
import numpy as np
import os





CARBON_FACTORS = {
    '\u5317\u4eac': 0.5554,
    '\u5929\u6d25': 0.6796,
    '\u6cb3\u5317': 0.6516,
    '\u5c71\u897f': 0.6634,
    '\u5185\u8499\u53e4': 0.6479,
    '\u8fbd\u5b81': 0.4878,
    '\u5409\u6797': 0.4671,
    '\u9ed1\u9f99\u6c5f': 0.5229,
    '\u4e0a\u6d77': 0.5737,
    '\u6c5f\u82cf': 0.5827,
    '\u6d59\u6c5f': 0.4974,
    '\u5b89\u5fbd': 0.6553,
    '\u798f\u5efa': 0.4211,
    '\u6c5f\u897f': 0.5836,
    '\u5c71\u4e1c': 0.6191,
    '\u6cb3\u5357': 0.5897,
    '\u6e56\u5317': 0.4044,
    '\u6e56\u5357': 0.4976,
    '\u5e7f\u4e1c': 0.4419,
    '\u5e7f\u897f': 0.4476,
    '\u6d77\u5357': 0.3648,
    '\u91cd\u5e86': 0.5581,
    '\u56db\u5ddd': 0.1564,
    '\u8d35\u5dde': 0.5683,
    '\u4e91\u5357': 0.1333,
    '\u9655\u897f': 0.6335,
    '\u7518\u8083': 0.4471,
    '\u9752\u6d77': 0.1796,
    '\u5b81\u590f': 0.6187,
    '\u65b0\u7586': 0.6021,
    '\u897f\u85cf': 0.5543,
}


EN_TO_CN = {
    'beijing': '\u5317\u4eac', 'tianjin': '\u5929\u6d25', 'hebei': '\u6cb3\u5317', 'shanxi_yi': '\u5c71\u897f',
    'shanxi_san': '\u9655\u897f', 'shandong': '\u5c71\u4e1c', 'neimeng': '\u5185\u8499\u53e4', 'liaoning': '\u8fbd\u5b81',
    'jilin': '\u5409\u6797', 'heilongjiang': '\u9ed1\u9f99\u6c5f', 'shanghai': '\u4e0a\u6d77', 'jiangsu': '\u6c5f\u82cf',
    'zhejiang': '\u6d59\u6c5f', 'anhui': '\u5b89\u5fbd', 'fujian': '\u798f\u5efa', 'jiangxi': '\u6c5f\u897f',
    'henan': '\u6cb3\u5357', 'hubei': '\u6e56\u5317', 'hunan': '\u6e56\u5357', 'chongqing': '\u91cd\u5e86',
    'sichuan': '\u56db\u5ddd', 'gansu': '\u7518\u8083', 'qinghai': '\u9752\u6d77', 'ningxia': '\u5b81\u590f',
    'xinjiang': '\u65b0\u7586', 'xizang': '\u897f\u85cf', 'guangdong': '\u5e7f\u4e1c', 'guangxi': '\u5e7f\u897f',
    'hainan': '\u6d77\u5357', 'guizhou': '\u8d35\u5dde', 'yunnan': '\u4e91\u5357',
}



BASE_DIR = str(DATA_ROOT / '6_Province_level_analysis/4_datacenter')
OUTPUT_BASE = str(DATA_ROOT / '6_Province_level_analysis/5_emission')


SCENARIOS = [
    ('scenario1_high', 'scenario1_high'),        
    ('scenario2_typical', 'scenario2_typical'),
    ('scenario3_flexible', 'scenario3_flexible'),
]


def extract_province_year(filename):
    'Extract province year.'
    for suffix in [
        '_scenario1_hourly_curtailment.xlsx',
        '_scenario2_typical.xlsx',
        '_scenario3_flexible.xlsx',
    ]:
        if filename.endswith(suffix):
            name = filename.replace(suffix, '')
            parts = name.split('_')
            year = None
            for p in parts:
                if p in ['2023', '2024', '2025']:
                    year = int(p)
                    break
            if year:
                province = '_'.join(parts[:parts.index(str(year))])
                return province, year
    return None, None


def calculate_carbon_emissions():
    'Calculate carbon emissions.'

    os.makedirs(OUTPUT_BASE, exist_ok=True)

    all_stats = []

    for subdir, output_subdir in SCENARIOS:
        source_folder = os.path.join(BASE_DIR, subdir)
        output_folder = os.path.join(OUTPUT_BASE, output_subdir)
        os.makedirs(output_folder, exist_ok=True)

        print(f'\n===== {subdir} =====')

        files = sorted([
            f for f in os.listdir(source_folder)
            if f.endswith('.xlsx')
            and not f.startswith('generate')
            and not f.startswith('~$')
        ])

        scenario_stats = []

        for fname in files:
            prov_en, year = extract_province_year(fname)
            if prov_en is None:
                continue

            prov_cn = EN_TO_CN.get(prov_en)
            if prov_cn is None:
                print(f'  [Skip] {fname}: No matching province')
                continue

            factor = CARBON_FACTORS.get(prov_cn)
            if factor is None:
                print(f'  [Skip] {fname}: No carbon emission factor')
                continue

            
            df = pd.read_excel(os.path.join(source_folder, fname))
            baseload_MW = df['baseload_MW'].values  # MW

            
            
            # baseload_MW -> baseload_kW = baseload_MW * 1000
            
            
            carbon_hourly_kg = baseload_MW * 1000 * factor  # kgCO2/h

            
            result_df = df.copy()
            result_df['carbon_emission_kgCO2_perh'] = carbon_hourly_kg
            result_df['carbon_factor_kgCO2_perkWh'] = factor

            output_fname = fname.replace('.xlsx', '_carbon_emission.xlsx')
            result_df.to_excel(
                os.path.join(output_folder, output_fname),
                index=False, engine='openpyxl'
            )

            # ============ SummarySummary ============
            total_baseload_GWh = baseload_MW.sum() / 1000
            total_carbon_t = carbon_hourly_kg.sum() / 1000  # tCO2
            total_carbon_kt = total_carbon_t / 1000  # ktCO2

            scenario_stats.append({
                'province': prov_en,
                'province_cn': prov_cn,
                'year': year,
                'carbon_factor_kgCO2_perkWh': factor,
                'total_baseload_GWh': total_baseload_GWh,
                'total_carbon_tCO2': total_carbon_t,
                'total_carbon_ktCO2': total_carbon_kt,
            })

            print(
                f'  {prov_en}_{year} {factor}, '
                f'baseload={total_baseload_GWh:.1f} GWh, '
                f'\u78b3\u6392\u653e={total_carbon_kt:.2f} ktCO2'
            )

        
        stats_df = pd.DataFrame(scenario_stats)
        stats_df = stats_df.sort_values(['year', 'province']).reset_index(drop=True)
        summary_path = os.path.join(output_folder, f'{output_subdir}_carbon_summary.xlsx')
        stats_df.to_excel(summary_path, index=False, engine='openpyxl')
        print(f'  Summary: {summary_path}')

        all_stats.append((subdir, stats_df))

    print(f'Processed {sum(len(s) for _, s in all_stats)} files.')

    return all_stats


if __name__ == '__main__':
    print('=' * 60)
    print('Carbon emissions calculation')
    print('=' * 60)
    calculate_carbon_emissions()